# 01 — Profiling de datos Bronze

Objetivo: inspeccionar la fuente Bronze inmutable y validar **únicamente la jerarquía de origen**. En este notebook no se realiza renombrado ni transformación hacia Silver.

Jerarquía correcta: **FCT → LIN/WKA → TU → PS (Process Step / operación) → STEP (subpaso) → resultado**.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('Raíz del proyecto:', PROJECT_ROOT)

In [ ]:
BRONZE_ROOT = PROJECT_ROOT / 'data' / 'bronze'
files = sorted(BRONZE_ROOT.glob('*.csv'))
if not files:
    raise FileNotFoundError(f'No CSV files found in {BRONZE_ROOT}')
BRONZE_FILE = files[0]
bronze = pd.read_csv(BRONZE_FILE, sep=';', index_col=False)
print(BRONZE_FILE.name, bronze.shape)
display(bronze.head())

## Schema fuente y completitud

In [ ]:
profile = pd.DataFrame({
    'dtype': bronze.dtypes.astype(str),
    'null_count': bronze.isna().sum(),
    'null_pct': bronze.isna().mean().mul(100).round(2),
    'nunique': bronze.nunique(dropna=True),
}).sort_index()
display(profile)

## Cobertura de la jerarquía industrial

`LIN` y `WKA` presentan actualmente una relación de reporting cercana a 1:1, mientras que `TU` aporta el nivel más granular de unidad de apriete/estación. `PS_Comment` es el nombre legible de la operación. `STEP_Number` representa un subpaso dentro del PS.

In [ ]:
hierarchy_fields = [
    'FCT_ID','FCT_Label','LIN_ID','LIN_Label','WKA_ID','WKA_Label',
    'TU_ID','TU_Name','PS_ID','PS_Number','PS_Comment',
    'STEP_ID','STEP_Number','STEP_Type'
]
hierarchy = pd.DataFrame({
    'field': hierarchy_fields,
    'nunique': [bronze[c].nunique(dropna=True) for c in hierarchy_fields],
    'examples': [', '.join(bronze[c].dropna().astype(str).unique()[:5]) for c in hierarchy_fields],
})
display(hierarchy)

In [ ]:
lin_wka = bronze[['LIN_ID','LIN_Label','WKA_ID','WKA_Label']].drop_duplicates()
tu = bronze[['LIN_Label','WKA_Label','TU_ID','TU_Name']].drop_duplicates().sort_values(['LIN_Label','TU_Name'])
ps = bronze[['TU_Name','PS_ID','PS_Number','PS_Comment']].drop_duplicates().sort_values(['TU_Name','PS_Number','PS_Comment'])
substeps = bronze[['PS_ID','PS_Comment','STEP_ID','STEP_Number','STEP_Type']].drop_duplicates().sort_values(['PS_ID','PS_Comment','STEP_Number'])
print('Pares LIN/WKA distintos:', len(lin_wka))
print('TUs distintas:', bronze['TU_ID'].nunique())
print('Descripciones PS distintas:', bronze['PS_Comment'].nunique())
print('Números STEP distintos:', sorted(bronze['STEP_Number'].dropna().unique().tolist()))
display(lin_wka)
display(tu.head(20))
display(ps.head(30))
display(substeps.head(30))

## Cobertura temporal

In [ ]:
ts = (bronze['RES_DateTime'].astype(str).str.replace(r' ([+-]\d{1,2})$', lambda m: f' {int(m.group(1)):+03d}:00', regex=True))
ts = pd.to_datetime(ts, format='%Y-%m-%d %H:%M:%S %z', errors='coerce', utc=True)
print('Timestamps válidos:', ts.notna().sum(), '/', len(ts))
print('Rango UTC:', ts.min(), '→', ts.max())

## Observación semántica clave

Los valores numéricos `PS_ID` / `STEP_ID` pueden reutilizarse bajo descripciones o tipos de operación distintos. Por ello, Gold utiliza una identidad de proceso sensible a la configuración en lugar de asumir que los identificadores por sí solos describen completamente la operación.

In [ ]:
reuse = (
    bronze.groupby(['TU_ID','PS_ID','STEP_ID'], dropna=False)
          .agg(ps_name_variants=('PS_Comment','nunique'), step_type_variants=('STEP_Type','nunique'))
          .reset_index()
)
display(reuse.query('ps_name_variants > 1 or step_type_variants > 1').head(30))